In [11]:
import json
import os
import shutil
import random

PROJECT_ROOT = os.getcwd()
FINETUNE_DIR = os.path.join(PROJECT_ROOT, "2_preprocessed_datasets", "bccd_finetune")

def coco_to_yolo(json_path, output_label_dir):
    os.makedirs(output_label_dir, exist_ok=True)
    with open(json_path, 'r') as f:
        data = json.load(f)

    images = {img['id']: img for img in data['images']}

    # Clear old files
    for f_name in os.listdir(output_label_dir):
        os.remove(os.path.join(output_label_dir, f_name))

    for ann in data['annotations']:
        img_info = images[ann['image_id']]
        img_w, img_h = img_info['width'], img_info['height']

        x_min, y_min, w, h = ann['bbox']
        x_center = (x_min + w / 2) / img_w
        y_center = (y_min + h / 2) / img_h
        norm_w = w / img_w
        norm_h = h / img_h

        cat_id = ann['category_id']
        line = f"{cat_id} {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}\n"

        txt_name = os.path.splitext(img_info['file_name'])[0] + '.txt'
        txt_path = os.path.join(output_label_dir, txt_name)
        with open(txt_path, 'a') as out_f:
            out_f.write(line)

# 1. Convert Train and Val
for split in ["train", "val"]:
    json_file = os.path.join(FINETUNE_DIR, "annotations", f"{split}.json")
    label_dir = os.path.join(FINETUNE_DIR, "labels", split)
    if os.path.exists(json_file):
        print(f"🔄 Converting {split.upper()} JSON to YOLO format...")
        coco_to_yolo(json_file, label_dir)
    else:
        print(f"⚠️ Notice: {split.upper()} JSON not found.")

# 2. Automatically Create Internal Test Split (10% from train)
images_train = os.path.join(FINETUNE_DIR, "images", "train")
labels_train = os.path.join(FINETUNE_DIR, "labels", "train")
images_test = os.path.join(FINETUNE_DIR, "images", "test")
labels_test = os.path.join(FINETUNE_DIR, "labels", "test")

os.makedirs(images_test, exist_ok=True)
os.makedirs(labels_test, exist_ok=True)

# Check if test split is already created to avoid re-splitting on multiple runs
if len(os.listdir(images_test)) == 0:
    print("✂️ Creating Internal Test Split (Moving 10% from Train to Test)...")
    all_train_imgs = [f for f in os.listdir(images_train) if f.endswith(('.jpg', '.png', '.jpeg'))]
    random.seed(42) # For reproducibility
    test_imgs = random.sample(all_train_imgs, int(len(all_train_imgs) * 0.10))

    for img in test_imgs:
        shutil.move(os.path.join(images_train, img), os.path.join(images_test, img))
        label_file = os.path.splitext(img)[0] + '.txt'
        if os.path.exists(os.path.join(labels_train, label_file)):
            shutil.move(os.path.join(labels_train, label_file), os.path.join(labels_test, label_file))
    print(f"✅ Successfully moved {len(test_imgs)} images to internal test set.")
else:
    print("✅ Internal test set already exists. Skipping split.")

🔄 Converting TRAIN JSON to YOLO format...
🔄 Converting VAL JSON to YOLO format...
✂️ Creating Internal Test Split (Moving 10% from Train to Test)...
✅ Successfully moved 76 images to internal test set.


In [12]:
import yaml

yaml_path = os.path.join(PROJECT_ROOT, "scripts", "finetune_blood_cell.yaml")

dataset_config = {
    'path': FINETUNE_DIR,
    'train': 'images/train',  
    'val': 'images/val',      
    'test': 'images/test',    # Now this points to the newly created test split
    'nc': 3,
    'names': {0: 'RBC', 1: 'WBC', 2: 'Platelet'}
}

os.makedirs(os.path.dirname(yaml_path), exist_ok=True)
with open(yaml_path, 'w') as f:
    yaml.dump(dataset_config, f, default_flow_style=False)

print(f"✅ YAML config created at: {yaml_path}")

✅ YAML config created at: /home/linux-hasan/projects/blood_cell_project/scripts/finetune_blood_cell.yaml


In [13]:
from ultralytics import RTDETR

print("🚀 Starting Fine-Tuning Phase...")

pretrained_weights = os.path.join(PROJECT_ROOT, "3_models", "stage1_pretrained", "weights", "best.pt")
model = RTDETR(pretrained_weights)
output_dir = os.path.join(PROJECT_ROOT, "3_models")

results = model.train(
    data=yaml_path,
    epochs=100,            
    patience=25,           
    imgsz=640,
    batch=8,               
    workers=4,
    project=output_dir,
    name='stage2_finetuned',
    device=0,
    
    # Heavy augmentations for Medical imaging
    lr0=0.001,             
    lrf=0.01,
    degrees=90.0,          
    fliplr=0.5,            
    flipud=0.5,            
    mosaic=1.0,            
    mixup=0.2              
)
print("✅ Fine-tuning completed successfully!")

🚀 Starting Fine-Tuning Phase...
New https://pypi.org/project/ultralytics/8.4.22 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.166 🚀 Python-3.12.3 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4070 SUPER, 12282MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/linux-hasan/projects/blood_cell_project/scripts/finetune_blood_cell.yaml, degrees=90.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.2, mode=train, model=/home/linux-has

train: Scanning /home/linux-hasan/projects/blood_cell_project/2_preprocessed_datasets/bccd_finetune/labels/train... 689 images, 0 backgrounds, 0 corrupt: 100%|██████████| 689/689 [00:00<00:00, 955.06it/s]

train: New cache created: /home/linux-hasan/projects/blood_cell_project/2_preprocessed_datasets/bccd_finetune/labels/train.cache
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 36.0±4.7 MB/s, size: 12.7 KB)


val: Scanning /home/linux-hasan/projects/blood_cell_project/2_preprocessed_datasets/bccd_finetune/labels/val.cache... 73 images, 0 backgrounds, 0 corrupt: 100%|██████████| 73/73 [00:00<?, ?it/s]


...

  0%|          | 0/87 [00:00<?, ?it/s]/home/linux-hasan/projects/envs/pt/lib/python3.12/site-packages/torch/autograd/graph.py:824: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:93.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
     66/100      6.86G     0.3638     0.4899     0.2088         40        640: 100%|██████████| 87/87 [00:28<00:00,  3.07it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 5/5 [00:01<00:00,  4.72it/s]

                   all         73        967      0.763      0.849      0.853      0.512


EarlyStopping: Training stopped early as no improvement observed in last 25 epochs. Best results observed at epoch 41, best model saved as best.pt.
To update EarlyStopping(patience=25) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

66 epochs completed in 0.579 hours.
Optimizer stripped from /home/linux-hasan/projects/blood_cell_project/3_models/stage2_finetuned/weights/last.pt, 66.1MB
Optimizer stripped from /home/linux-hasan/projects/blood_cell_project/3_models/stage2_finetuned/weights/best.pt, 66.1MB

Validating /home/linux-hasan/projects/blood_cell_project/3_models/stage2_finetuned/weights/best.pt...
Ultralytics 8.3.166 🚀 Python-3.12.3 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4070 SUPER, 12282MiB)
rt-detr-l summary: 302 layers, 31,989,905 parameters, 0 gradients, 103.4 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 5/5 [00:01<00:00,  3.01it/s]


                   all         73        967      0.801      0.875      0.876      0.599
                   RBC         72        819      0.609      0.849      0.803      0.545
                   WBC         71         72      0.969          1      0.983      0.764
              Platelet         42         76      0.825      0.776      0.843      0.488
Speed: 0.1ms preprocess, 14.2ms inference, 0.0ms loss, 0.4ms postprocess per image
Results saved to /home/linux-hasan/projects/blood_cell_project/3_models/stage2_finetuned
✅ Fine-tuning completed successfully!


In [14]:
print("📊 Generating Paper-Ready Validation Metrics...")

finetuned_weights = os.path.join(PROJECT_ROOT, "3_models", "stage2_finetuned", "weights", "best.pt")
best_model = RTDETR(finetuned_weights)

val_metrics = best_model.val(
    data=yaml_path,
    split='val',
    project=output_dir,
    name='stage2_eval_val',
    save_json=True,       
    plots=True            
)
print("✅ Validation complete.")

📊 Generating Paper-Ready Validation Metrics...
Ultralytics 8.3.166 🚀 Python-3.12.3 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4070 SUPER, 12282MiB)
rt-detr-l summary: 302 layers, 31,989,905 parameters, 0 gradients, 103.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 922.9±232.3 MB/s, size: 12.9 KB)


val: Scanning /home/linux-hasan/projects/blood_cell_project/2_preprocessed_datasets/bccd_finetune/labels/val.cache... 73 images, 0 backgrounds, 0 corrupt: 100%|██████████| 73/73 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 5/5 [00:02<00:00,  2.35it/s]


                   all         73        967      0.801      0.874      0.877      0.601
                   RBC         72        819       0.61      0.846      0.803      0.546
                   WBC         71         72      0.969          1      0.983      0.767
              Platelet         42         76      0.826      0.776      0.843      0.489
Speed: 2.7ms preprocess, 17.5ms inference, 0.0ms loss, 0.7ms postprocess per image
Saving /home/linux-hasan/projects/blood_cell_project/3_models/stage2_eval_val/predictions.json...
Results saved to /home/linux-hasan/projects/blood_cell_project/3_models/stage2_eval_val
✅ Validation complete.


In [15]:
print("🧪 Running Rigorous Internal Testing...")

test_metrics = best_model.val(
    data=yaml_path,
    split='test',
    project=output_dir,
    name='stage2_eval_internal_test',
    save_json=True,       
    plots=True            
)
print("✅ Internal Testing complete! Check '3_models/stage2_eval_internal_test' for final results.")

🧪 Running Rigorous Internal Testing...
Ultralytics 8.3.166 🚀 Python-3.12.3 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4070 SUPER, 12282MiB)
rt-detr-l summary: 302 layers, 31,989,905 parameters, 0 gradients, 103.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 47.0±17.0 MB/s, size: 13.2 KB)


val: Scanning /home/linux-hasan/projects/blood_cell_project/2_preprocessed_datasets/bccd_finetune/labels/test... 76 images, 0 backgrounds, 0 corrupt: 100%|██████████| 76/76 [00:00<00:00, 999.71it/s]

val: New cache created: /home/linux-hasan/projects/blood_cell_project/2_preprocessed_datasets/bccd_finetune/labels/test.cache



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 5/5 [00:02<00:00,  2.41it/s]


                   all         76       1021      0.769      0.925      0.896      0.623
                   RBC         72        873      0.564      0.948      0.838      0.606
                   WBC         74         74       0.96          1      0.995      0.832
              Platelet         40         74      0.782      0.825      0.857      0.431
Speed: 2.4ms preprocess, 17.0ms inference, 0.0ms loss, 0.5ms postprocess per image
Saving /home/linux-hasan/projects/blood_cell_project/3_models/stage2_eval_internal_test/predictions.json...
Results saved to /home/linux-hasan/projects/blood_cell_project/3_models/stage2_eval_internal_test
✅ Internal Testing complete! Check '3_models/stage2_eval_internal_test' for final results.


In [16]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

results_path = os.path.join(PROJECT_ROOT, "3_models", "stage2_finetuned", "results.csv")

if os.path.exists(results_path):
    df = pd.read_csv(results_path)
    df.columns = df.columns.str.strip()

    # Find the right loss columns dynamically
    train_loss_col = [c for c in df.columns if 'train' in c and ('giou_loss' in c or 'box_loss' in c)][0]
    val_loss_col = [c for c in df.columns if 'val' in c and ('giou_loss' in c or 'box_loss' in c)][0]

    plt.figure(figsize=(10, 6))
    sns.set_theme(style="whitegrid")
    
    sns.lineplot(data=df, x='epoch', y=train_loss_col, label='Train Loss', linewidth=2, color='darkblue')
    sns.lineplot(data=df, x='epoch', y=val_loss_col, label='Validation Loss', linewidth=2, color='darkorange')
    
    plt.title('Training vs Validation Loss', fontsize=14, fontweight='bold')
    plt.xlabel('Epochs', fontsize=12)
    plt.ylabel('Loss Value', fontsize=12)
    plt.legend()
    sns.despine()
    
    plot_path = os.path.join(PROJECT_ROOT, "3_models", "stage2_finetuned", "custom_train_val_curve.png")
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✅ Loss curve successfully saved using seaborn at: {plot_path}")
else:
    print("⏳ results.csv not found.")

<Figure size 1000x600 with 1 Axes>

✅ Loss curve successfully saved using seaborn at: /home/linux-hasan/projects/blood_cell_project/3_models/stage2_finetuned/custom_train_val_curve.png


In [19]:
import os

print("🌍 Running Inference on External Dataset (wbc_test)...")

# Screenshot onujayi apnar exact path (amra 'val' folder er chobi gulor upor test korchi)
EXTERNAL_TEST_DIR = os.path.join(PROJECT_ROOT, "2_preprocessed_datasets", "wbc_test", "images", "val")

print(f"📂 Looking for images exactly in: {EXTERNAL_TEST_DIR}")

# Inference run kora
if os.path.exists(EXTERNAL_TEST_DIR):
    external_results = best_model.predict(
        source=EXTERNAL_TEST_DIR,
        conf=0.40,            
        iou=0.50,             
        project=output_dir,
        name='stage2_external_inference',
        save=True,            
        save_txt=True         
    )
    print("✅ External Testing complete. Visuals saved!")
else:
    print(f"❌ Error: Folder ta ekhono pawa jacche na: {EXTERNAL_TEST_DIR}")

🌍 Running Inference on External Dataset (wbc_test)...
📂 Looking for images exactly in: /home/linux-hasan/projects/blood_cell_project/2_preprocessed_datasets/wbc_test/images/val

WARNING ⚠️ 
inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/1931 /home/linux-hasan/projects/blood_cell_project/2_preprocessed_datasets/wbc_test/images/val/20190526_163027_0_jpg.rf.943b0a8a4e56431ce7ec29fb97ea5227.jpg: 640x640 1 RBC, 2 WBCs, 41.2ms
image 2/1931 /home/linux-hasan/projects/blood_cell_project/2_preprocessed_datasets/

In [20]:
print("📦 Exporting Final Fine-Tuned Model...")

exported_path = best_model.export(
    format='onnx', 
    dynamic=True,         
    simplify=True         
)

print(f"✅ Model exported successfully to: {exported_path}")
print("🎉 The Complete Fine-Tuning Pipeline is successfully finished!")

📦 Exporting Final Fine-Tuned Model...
Ultralytics 8.3.166 🚀 Python-3.12.3 torch-2.7.1+cu118 CPU (AMD Ryzen 7 7700 8-Core Processor)
rt-detr-l summary: 302 layers, 31,989,905 parameters, 0 gradients, 103.4 GFLOPs

PyTorch: starting from '/home/linux-hasan/projects/blood_cell_project/3_models/stage2_finetuned/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 7) (63.1 MB)

ONNX: starting export with onnx 1.17.0 opset 19...
ONNX: slimming with onnxslim 0.1.59...
ONNX: export success ✅ 17.0s, saved as '/home/linux-hasan/projects/blood_cell_project/3_models/stage2_finetuned/weights/best.onnx' (122.5 MB)

Export complete (18.4s)
Results saved to /home/linux-hasan/projects/blood_cell_project/3_models/stage2_finetuned/weights
Predict:         yolo predict task=detect model=/home/linux-hasan/projects/blood_cell_project/3_models/stage2_finetuned/weights/best.onnx imgsz=640  
Validate:        yolo val task=detect model=/home/linux-hasan/projects/blood_cell_projec